# Chai-1 Structure Prediction

This notebook demonstrates multi-modal structure prediction using Chai-1 from Chai Discovery. Chai-1 predicts the three-dimensional structures of proteins, ligands, and glycans, as well as their complexes, by combining ESM language model embeddings with a trunk network and diffusion-based structure generation. We demonstrate `run_chai1` by predicting the structure of the MfnG protein bound to L-tyrosine, covering input construction, model configuration, result inspection, visualization, and export.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("chai1")
display_overview("chai1")
display_docs_section("chai1", "Background")

# Chai1

> Chai1 is a multi-modal structure prediction model from Chai Discovery that predicts 3D structures of proteins, ligands, and glycans using a [diffusion](https://en.wikipedia.org/wiki/Diffusion_model)-based architecture. It excels at modeling protein-ligand complexes and can incorporate ESM embeddings for improved accuracy.

## Background

**What does this tool measure/predict?**
Chai1 predicts the 3D atomic coordinates of biomolecular complexes from sequences. It outputs full-atom structures for proteins, ligands, and glycans with confidence scores including per-residue pLDDT, interface pTM (ipTM), and overall structure confidence.

**Why is this important?**
Understanding how proteins interact with small molecules is critical for:

* Drug design and virtual screening (predicting binding poses)
* Validating designed protein-ligand interactions
* Modeling glycoprotein structures with attached carbohydrates
* Predicting binding site conformations
* Screening compound libraries against protein targets

**Scientific foundation:**
Chai1 uses a modern deep learning architecture combining:

1. **ESM-2 embeddings** (optional): Pre-trained protein language model representations that capture evolutionary and structural patterns from millions of protein sequences.
2. **Diffusion-based structure generation**: A generative modeling approach that iteratively denoises random coordinates into realistic 3D structures, naturally handling the multi-modal nature of biomolecular complexes.
3. **Trunk network**: An attention-based architecture that processes sequence features through multiple recycling passes for iterative refinement.

Confidence metrics include:

* **pLDDT** (predicted Local Distance Difference Test): Per-residue confidence score (0-100), where >90 indicates high confidence, 70-90 is moderate, and \<70 suggests low confidence.
* **pTM** (predicted Template Modeling score): Overall structure accuracy (0-1), where >0.8 indicates high confidence in the global fold.
* **ipTM** (interface pTM): Confidence in inter-chain interfaces (0-1), critical for multi-chain complexes.

## Available tools

In [2]:
display_available_tools("chai1")

- **`run_chai1()`** — Multi-modal structure prediction using Chai1

### `run_chai1`

Chai-1 predicts 3D structures of proteins, ligands, and glycans using a diffusion-based model that optionally integrates ESM protein language model embeddings for improved predictions without requiring an explicit MSA search. The `num_trunk_recycles` parameter controls iterative refinement of the structure representation, while `num_diffn_timesteps` governs the resolution of the denoising process. Multiple independent diffusion samples can be generated via `num_diffn_samples`, with the best sample returned by confidence score. Chai-1 has a maximum total sequence length of 2,048 residues per complex and does not support DNA or RNA inputs.

In [3]:
from pathlib import Path

from proto_tools import (
    Chai1Config,
    Chai1Input,
    Chain,
    StructurePredictionComplex,
    run_chai1,
)

In [4]:
# Display input docs
display_api_reference("chai1", "input", "run_chai1")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = Chai1Input(complexes=[complex])

**Input** — `Chai1Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | required | List of complexes to predict structures for. Inherited from `StructurePredictionInput`. Each complex can contain multiple chains of proteins, ligands, and/or glycans. Total length across all chains in a complex must not exceed 2,048 residues. |
| `chains` | `List[Chain]` | required | Chains in the complex. Each chain can be: |
| `msas` | `Dict[string, MSA]` |  | Pre-computed MSAs keyed by protein sequence. Populated by preprocess() or supplied directly. Default: None. |

In [5]:
# Display config docs
display_api_reference("chai1", "config", "run_chai1")

# Configure Chai-1
config = Chai1Config(
    verbose=True,
    device="cuda",  # Change to "cpu" if no GPU available
    use_esm_embeddings=True,
    use_msa=False,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    num_diffn_samples=1,
)

**Config** — `Chai1Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `use_esm_embeddings` | `boolean` | `True` | Whether to use ESM (Evolutionary Scale Modeling) embeddings for improved predictions. ESM embeddings provide evolutionary context from large-scale protein language models, typically improving prediction quality. Default: `True`. |
| `num_trunk_recycles` | `integer` | `3` | Number of iterative refinement passes through the trunk network. Higher values produce more refined structures but increase computation time. Typical range: 0-10. Must be at least 0. |
| `num_diffn_timesteps` | `integer` | `200` | Number of denoising steps in the diffusion process. Higher values produce more refined structures but are slower. Typical |
| `num_diffn_samples` | `integer` | `1` | Number of independent structure samples to generate per complex via the diffusion process. Only the best sample (by confidence) is returned. Higher values explore more conformational space but increase computation time. Must be at least 1. Default: 1. |
| `num_trunk_samples` | `integer` | `1` | Number of independent trunk forward passes per diffusion sample. Increases diversity in structure generation. Must be at least 1. Default: 1. |
| `use_msa` | `boolean` | `True` | Whether to generate and use Multiple Sequence Alignments (MSAs) for protein chains using ColabFold search. Inherited from `MSAStructurePredictionConfig`. Default: `True`. |
| `seed` | `integer` | `42` | Random seed for reproducible results. Set to a fixed value for deterministic predictions or `None` for random behavior. |
| `verbose` | `boolean` | `False` | Whether to print status messages during execution. Inherited from `StructurePredictionConfig`. Default: `False`. |
| `device` | `string` | `cuda` | Device to run the model on (`"cuda"`, `"cpu"`). Inherited from `StructurePredictionConfig`. Default: `"cuda"`. |
| `timeout` | `integer` | `1200` | Maximum execution time in seconds. Default: 1200. |
| `colabfold_search_config` | `ColabfoldSearchConfig` |  | Configuration for ColabFold MSA search. Only used when `use_msa=True`. Inherited from `MSAStructurePredictionConfig`. Default: `None`. |

In [6]:
# Run structure prediction
result = run_chai1(inputs, config)

Folding structures (Chai-1):   0%|          | 0/1 [00:00<?, ?complex/s]

Running inference.py (one-shot) with device=cuda:0
/raid/projects/viggiano/codebases/evo-design/proto_tool_envs/chai1_env/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:154: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)
Diffusion steps: 100%|██████████| 199/199 [00:37<00:00,  

Score=0.6849, writing output to /tmp/tmpjbnoo1nz/output/pred.model_idx_0.cif


In [7]:
# Display output docs
display_api_reference("chai1", "output", "run_chai1")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Average pLDDT: {mfng_structure.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.iptm:.3f}")

**Output** — `StructurePredictionOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `List[object]` | required | List of predicted structures, one per input complex. Each structure contains the 3D coordinates in CIF format along with model-specific confidence metrics. The order matches the input complexes order. |

  Number of chains: 2
  Protein length: 384 residues
  Average pLDDT: 0.837
  pTM score: 0.847
  ipTM score: 0.644


#### Visualize the predicted complex

The interactive viewer renders the predicted protein-ligand complex colored by chain. You can rotate, zoom, and inspect the binding interface between MfnG and L-tyrosine.

In [10]:
mfng_structure.visualize(color_by='chain')

## Export Results

Predicted structures can be exported to PDB or mmCIF format for further analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT confidence scores for per-residue visualization.

In [9]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'mfng_ligand_complex.pdb'}")

Structure exported to example_output/mfng_ligand_complex.pdb


/raid/projects/viggiano/codebases/evo-design/proto-tools/proto_tools/entities/structures/structure.py:467: UserWarning: CIF→PDB conversion: 1 residue name(s) exceed PDB's 3-character limit and will be truncated (e.g., ['LIG2']).
  return convert_cif_str_to_pdb_str(self.structure)
